In [1]:
import torch
import pandas as pd
import numpy as np
#import random
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold,StratifiedKFold 
import gc
import os
from os import path
from sys import path as systemPath
systemPath.append(path.join('..', '..'))
from scipy.spatial import distance_matrix
#import argparse
import random
import matplotlib.pyplot as plt
import networkx as nx
import MDAnalysis as mda
from matplotlib.ticker import MaxNLocator,MultipleLocator
from MDAnalysis.core.groups import AtomGroup
import seaborn as sns
import statistics as st

In [2]:
##儲存資料
radius_all=np.load(f'D:/collagen/new_uh_rad/traj_radius_all.npy', allow_pickle = True)
radius_wt=np.load(f'D:/collagen/new_uh_rad/traj_radius_wt.npy', allow_pickle = True)
unit_hei_all=np.load(f'D:/collagen/new_uh_rad/traj_unit_hei.npy', allow_pickle = True)
unit_hei_wt=np.load(f'D:/collagen/new_uh_rad/traj_unit_hei_wt.npy', allow_pickle = True)

gpo_radius_traj = np.load(f'D:/collagen/gpo30/ana/gpo_traj_radius.npy', allow_pickle = True)
gpo_unit_hei_traj = np.load(f'D:/collagen/gpo30/ana/gpo_traj_unhei.npy', allow_pickle = True)

mutation_name_arr = np.load(f'D:/collagen/data/mutation_name.npy')  # (587,)
lethal_arr = np.load(f'D:/collagen/data/lethal.npy')  # (587,)
alpha12_arr = np.load(f'D:/collagen/data/alpha12.npy')  # (587,)
triplet_number_arr = np.load(f'D:/collagen/data/triplet_number.npy')  # (587,)

In [16]:
# real position index
mutation_pos = [2069]
# mutation type
mutation_name = ["ALA"]
# lethal:0, non-lethal:1 
lethal = [1]
# triple number 
triplet_number = [336]
# chain 
alpha12 = [2]

In [22]:
import numpy as np
from scipy import interpolate

def compute_collagen_radius(center_points, ca_positions, smooth=0.5, n_axis_points=200, mode='natural'):
    """
    計算膠原蛋白三股螺旋在單一 frame 下的 radius profile。
    可選擇使用 'parametric' 或 'natural' 三次樣條。

    Parameters
    ----------
    center_points : np.ndarray, shape (N, 3)
        每個 Gly 層 (i-th position) 的中心點 r_i = (r_iA + r_iB + r_iC)/3。
    ca_positions : np.ndarray, shape (N, 3, 3)
        每個位置三股鏈的 Cα 原子座標：
        ca_positions[i,0] = chain A 的 Cα at position i
        ca_positions[i,1] = chain B 的 Cα at position i
        ca_positions[i,2] = chain C 的 Cα at position i
    smooth : float
        僅對 parametric 模式有效。平滑參數 s（越大越平滑）。
    n_axis_points : int
        樣條曲線插值點數。
    mode : {'parametric', 'natural'}
        样條模式：
        - 'parametric' 使用 splprep (支援平滑)
        - 'natural' 使用 CubicSpline (嚴格插值)

    Returns
    -------
    radius_profile : np.ndarray, shape (N,)
        每個位置的平均半徑。
    axis_points : np.ndarray, shape (n_axis_points, 3)
        樣條軸線的 3D 座標，用於畫圖。
    """

    N = len(center_points)

    # ----------------------------------------------------------
    # STEP 1. 建立樣條軸線
    # ----------------------------------------------------------
    if mode == 'parametric':
        # --- 使用三維參數化樣條 (splprep)
        x, y, z = center_points[:, 0], center_points[:, 1], center_points[:, 2]
        tck, u = interpolate.splprep([x, y, z], s=smooth, k=3)
        u_fine = np.linspace(0, 1, n_axis_points)
        x_axis, y_axis, z_axis = interpolate.splev(u_fine, tck)
        axis_points = np.vstack([x_axis, y_axis, z_axis]).T

    elif mode == 'natural':
        # --- 使用三維自然樣條 (CubicSpline)
        # 自行建立參數軸（相當於點的順序）
        u = np.linspace(0, 1, N)
        cs_x = interpolate.CubicSpline(u, center_points[:, 0], bc_type='natural')
        cs_y = interpolate.CubicSpline(u, center_points[:, 1], bc_type='natural')
        cs_z = interpolate.CubicSpline(u, center_points[:, 2], bc_type='natural')

        u_fine = np.linspace(0, 1, n_axis_points)
        x_axis, y_axis, z_axis = cs_x(u_fine), cs_y(u_fine), cs_z(u_fine)
        axis_points = np.vstack([x_axis, y_axis, z_axis]).T

    else:
        raise ValueError("mode must be 'parametric' or 'natural'")

    # ----------------------------------------------------------
    # STEP 2. 計算每層 Gly 到軸的距離 (radius)
    # ----------------------------------------------------------
    radius_profile = []
    for i in range(N):
        radii_i = []
        for chain in range(3):
            atom_pos = ca_positions[i, chain]  # shape (3,)
            diffs = axis_points - atom_pos
            dist = np.min(np.linalg.norm(diffs, axis=1))
            radii_i.append(dist)
        radius_profile.append(np.mean(radii_i))

    return np.array(radius_profile), axis_points
radius_all = []
for tri_pos, mpos, mtype, l, a12 in zip(triplet_number, mutation_pos, mutation_name, lethal, alpha12):
    # 以 tri_pos 當作 center（也可以改用 mutation_pos）
    center = tri_pos  
    # 建立 mutation ID（方便除錯與後續標籤）
    mutation_id = f"{tri_pos}_{mtype.upper()}_{a12}"
    
    # 依據公式計算各條鏈的基準殘基編號：
    # 原本：x1 = 3*(center-1)+17
    x1 = 3 * (center - 1) + 17
    x2 = 3 * (center - 1) + 1054 + 10
    x3 = 3 * (center - 1) + 17 + 2080
    
    # 計算前4後4共9個位置（原始程式用 range(-12,13,3)）
    # 這9個數值對應於相對位置 P12, P9, P6, P3, M0, P3', P6', P9', P12'
    x1_values = [x1 + i for i in range(-12, 13, 3)]
    x2_values = [x2 + i for i in range(-12, 13, 3)]
    x3_values = [x3 + i for i in range(-12, 13, 3)]
    
    # ----------------------------
    # mutant 結構計算
    # ----------------------------
    
    file_psf = f"D:/collagen/a{a12}/{tri_pos}_{mtype.upper()}_{a12}/mutation_{tri_pos}_{mtype.upper()}__{l}_{a12}.psf"
    file_pdb = f"D:/collagen/a{a12}_dcd_cut/{tri_pos}_{mtype.upper()}.dcd"
    u1 = mda.Universe(file_psf, file_pdb)

    radius_traj = [[] for _ in range(100)]
    for t in u1.trajectory:
        if a12 == 1:
            alpha1_1_mu = u1.select_atoms(f"segid 0A and resid {x1_values[4]}")  # 中心位置 index 4
            alpha1_2_mu = u1.select_atoms(f"segid 0C and resid {x3_values[4]}")
            alpha2_mu   = u1.select_atoms(f"segid 0B and resid {x2_values[4]}")
            if alpha1_1_mu.resnames[0] != mtype.upper() or alpha1_2_mu.resnames[0] != mtype.upper():
                print(f"Warning: Mutation {mutation_id} chain 0A or 0C not matching {mtype.upper()}")
            if alpha2_mu.resnames[0] != "GLY":
                print(f"Warning: Mutation {mutation_id} chain 0B is not GLY")
        elif a12 == 2:
            alpha2_mu   = u1.select_atoms(f"segid 0B and resid {x2_values[4]}")
            alpha1_1_mu = u1.select_atoms(f"segid 0A and resid {x1_values[4]}")
            alpha1_2_mu = u1.select_atoms(f"segid 0C and resid {x3_values[4]}")
            if alpha2_mu.resnames[0] != mtype.upper():
                print(f"Warning: Mutation {mutation_id} chain 0B not matching {mtype.upper()}")
            if alpha1_1_mu.resnames[0] != "GLY" or alpha1_2_mu.resnames[0] != "GLY":
                print(f"Warning: Mutation {mutation_id} chain 0A or 0C is not GLY")
        
        # 計算 mutant 的半徑 profile：對9個位置逐一計算
        radius_profile_mut = []  # mutant 的半徑 profile
        combined_center_list = []
        ca_positions = []
        for r1, r2, r3 in zip(x1_values, x2_values, x3_values):
            # 分別選取 mutant 結構中各鏈該位置的原子
            a1 = u1.select_atoms(f"segid 0A and resid {r1}")
            a2 = u1.select_atoms(f"segid 0B and resid {r2}")
            a3 = u1.select_atoms(f"segid 0C and resid {r3}")
            print(f"Frame {t.frame}: Selecting resid {r1} (chain 0A), {r2} (chain 0B), {r3} (chain 0C)")
            # 檢查是否有空 selection
            if len(a1) == 0 or len(a2) == 0 or len(a3) == 0:
                print(f"⚠️ Warning: empty selection at frame {t.frame} | resid: {r1}, {r2}, {r3}")
                continue  # 或用 np.nan 取代
            # 分別計算各原子群的幾何中心
            c1 = a1.center_of_geometry()
            c2 = a2.center_of_geometry()
            c3 = a3.center_of_geometry()
            # 三條鏈的 Cα 坐標
            ca_positions.append(np.array([c1, c2, c3]))
            print(np.array(ca_positions).shape)
            # 合併三個原子群，計算整體幾何中心
            combined = AtomGroup(a1 + a2 + a3)
            combined_center = combined.center_of_geometry()
            combined_center_list.append(combined_center)
        combined_center_list = np.array(combined_center_list) #shape (9, 3) 9 positions，每個位置3D座標(x,y,z)
        ca_positions = np.array(ca_positions)  # shape (9, 3, 3) 9 positions，每個位置3條鏈的Cα的3D座標
        radius_profile_mut, axis_points = compute_collagen_radius(
            combined_center_list,
            ca_positions,
            smooth=0.5
        )
        # print(radius_profile_mut)
        radius_traj [t.frame] = radius_profile_mut

    radius_all.append(radius_traj)
radius_all = np.array (radius_all)

c:\Users\wendy\anaconda3\Lib\site-packages\MDAnalysis\coordinates\DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


Frame 0: Selecting resid 1010 (chain 0A), 2057 (chain 0B), 3090 (chain 0C)
(1, 3, 3)
Frame 0: Selecting resid 1013 (chain 0A), 2060 (chain 0B), 3093 (chain 0C)
(2, 3, 3)
Frame 0: Selecting resid 1016 (chain 0A), 2063 (chain 0B), 3096 (chain 0C)
(3, 3, 3)
Frame 0: Selecting resid 1019 (chain 0A), 2066 (chain 0B), 3099 (chain 0C)
(4, 3, 3)
Frame 0: Selecting resid 1022 (chain 0A), 2069 (chain 0B), 3102 (chain 0C)
(5, 3, 3)
Frame 0: Selecting resid 1025 (chain 0A), 2072 (chain 0B), 3105 (chain 0C)
(6, 3, 3)
Frame 0: Selecting resid 1028 (chain 0A), 2075 (chain 0B), 3108 (chain 0C)
(7, 3, 3)
Frame 0: Selecting resid 1031 (chain 0A), 2078 (chain 0B), 3111 (chain 0C)
(8, 3, 3)
Frame 0: Selecting resid 1034 (chain 0A), 2081 (chain 0B), 3114 (chain 0C)
⚠️ Warning: empty selection at frame 0 | resid: 1034, 2081, 3114
Frame 1: Selecting resid 1010 (chain 0A), 2057 (chain 0B), 3090 (chain 0C)
(1, 3, 3)
Frame 1: Selecting resid 1013 (chain 0A), 2060 (chain 0B), 3093 (chain 0C)
(2, 3, 3)
Frame 1: S

In [48]:
import numpy as np
from scipy import interpolate

def compute_collagen_radius(center_points, ca_positions, smooth=0.5, n_axis_points=200, mode='natural'):
    """
    計算膠原蛋白三股螺旋在單一 frame 下的 radius profile。
    可選擇使用 'parametric' 或 'natural' 三次樣條。

    Parameters
    ----------
    center_points : np.ndarray, shape (N, 3)
        每個 Gly 層 (i-th position) 的中心點 r_i = (r_iA + r_iB + r_iC)/3。
    ca_positions : np.ndarray, shape (N, 3, 3)
        每個位置三股鏈的 Cα 原子座標：
        ca_positions[i,0] = chain A 的 Cα at position i
        ca_positions[i,1] = chain B 的 Cα at position i
        ca_positions[i,2] = chain C 的 Cα at position i
    smooth : float
        僅對 parametric 模式有效。平滑參數 s（越大越平滑）。
    n_axis_points : int
        樣條曲線插值點數。
    mode : {'parametric', 'natural'}
        样條模式：
        - 'parametric' 使用 splprep (支援平滑)
        - 'natural' 使用 CubicSpline (嚴格插值)

    Returns
    -------
    radius_profile : np.ndarray, shape (N,)
        每個位置的平均半徑。
    axis_points : np.ndarray, shape (n_axis_points, 3)
        樣條軸線的 3D 座標，用於畫圖。
    """

    N = len(center_points)

    # ----------------------------------------------------------
    # STEP 1. 建立樣條軸線
    # ----------------------------------------------------------
    if mode == 'parametric':
        # --- 使用三維參數化樣條 (splprep)
        x, y, z = center_points[:, 0], center_points[:, 1], center_points[:, 2]
        tck, u = interpolate.splprep([x, y, z], s=smooth, k=3)
        u_fine = np.linspace(0, 1, n_axis_points)
        x_axis, y_axis, z_axis = interpolate.splev(u_fine, tck)
        axis_points = np.vstack([x_axis, y_axis, z_axis]).T

    elif mode == 'natural':
        # --- 使用三維自然樣條 (CubicSpline)
        # 自行建立參數軸（相當於點的順序）
        u = np.linspace(0, 1, N)
        cs_x = interpolate.CubicSpline(u, center_points[:, 0], bc_type='natural')
        cs_y = interpolate.CubicSpline(u, center_points[:, 1], bc_type='natural')
        cs_z = interpolate.CubicSpline(u, center_points[:, 2], bc_type='natural')

        u_fine = np.linspace(0, 1, n_axis_points)
        x_axis, y_axis, z_axis = cs_x(u_fine), cs_y(u_fine), cs_z(u_fine)
        axis_points = np.vstack([x_axis, y_axis, z_axis]).T

    else:
        raise ValueError("mode must be 'parametric' or 'natural'")

    # ----------------------------------------------------------
    # STEP 2. 計算每層 Gly 到樣條軸的距離 (radius)
    # ----------------------------------------------------------
    radius_profile = []

    for i in range(N):  # N 層 (通常 9)
        radii_i = []    # 存放這一層三條鏈的半徑

        # --- 逐條鏈計算半徑 ---
        for chain in range(3):
            atom_pos = ca_positions[i, chain]  # 該層該鏈的 Cα 原子座標 (x, y, z)

            # 若該鏈缺失 (例如為 [nan, nan, nan])，跳過這條鏈
            if np.isnan(atom_pos).any():
                print(f"⚠️ Layer {i}, Chain {chain} is missing (skipped)")
                continue

            # 計算此原子到樣條軸所有點的距離
            diffs = axis_points - atom_pos                  # 每個樣條點到此原子的向量
            dist_all = np.linalg.norm(diffs, axis=1)        # 距離陣列
            dist_min = np.min(dist_all)                     # 最短距離 = 半徑
            radii_i.append(dist_min)

        # --- 對該層的三條鏈半徑取平均 ---
        # 若有鏈缺失 (radii_i 可能少於 3)，只對現有值取平均
        if len(radii_i) > 0:
            avg_radius = np.mean(radii_i)
        else:
            avg_radius = 0.0  # 若三條鏈都缺失，半徑設為 0

        # 儲存這層的平均半徑
        radius_profile.append(avg_radius)

    return np.array(radius_profile), axis_points
# 計算每一種突變最後100個frame每個frame每個位置的radius
radius_all = []
for tri_pos, mpos, mtype, l, a12 in zip(triplet_number, mutation_pos, mutation_name, lethal, alpha12):
    # 以 tri_pos 當作 center（也可以改用 mutation_pos）
    center = tri_pos  
    # 建立 mutation ID（方便除錯與後續標籤）
    mutation_id = f"{tri_pos}_{mtype.upper()}_{a12}"
    
    # 依據公式計算各條鏈的基準殘基編號：
    # 原本：x1 = 3*(center-1)+17
    x1 = 3 * (center - 1) + 17
    x2 = 3 * (center - 1) + 1054 + 10
    x3 = 3 * (center - 1) + 17 + 2080
    
    # 計算前4後4共9個位置（原始程式用 range(-12,13,3)）
    # 這9個數值對應於相對位置 P12, P9, P6, P3, M0, P3', P6', P9', P12'
    x1_values = [x1 + i for i in range(-12, 13, 3)]
    x2_values = [x2 + i for i in range(-12, 13, 3)]
    x3_values = [x3 + i for i in range(-12, 13, 3)]
    
    # ----------------------------
    # mutant 結構計算
    # ----------------------------
    
    file_psf = f"D:/collagen/a{a12}/{tri_pos}_{mtype.upper()}_{a12}/mutation_{tri_pos}_{mtype.upper()}__{l}_{a12}.psf"
    file_pdb = f"D:/collagen/a{a12}_dcd_cut/{tri_pos}_{mtype.upper()}.dcd"
    u1 = mda.Universe(file_psf, file_pdb)

    radius_traj = [[] for _ in range(100)]
    for t in u1.trajectory:
        if a12 == 1:
            alpha1_1_mu = u1.select_atoms(f"segid 0A and resid {x1_values[4]}")  # 中心位置 index 4
            alpha1_2_mu = u1.select_atoms(f"segid 0C and resid {x3_values[4]}")
            alpha2_mu   = u1.select_atoms(f"segid 0B and resid {x2_values[4]}")
            if alpha1_1_mu.resnames[0] != mtype.upper() or alpha1_2_mu.resnames[0] != mtype.upper():
                print(f"Warning: Mutation {mutation_id} chain 0A or 0C not matching {mtype.upper()}")
            if alpha2_mu.resnames[0] != "GLY":
                print(f"Warning: Mutation {mutation_id} chain 0B is not GLY")
        elif a12 == 2:
            alpha2_mu   = u1.select_atoms(f"segid 0B and resid {x2_values[4]}")
            alpha1_1_mu = u1.select_atoms(f"segid 0A and resid {x1_values[4]}")
            alpha1_2_mu = u1.select_atoms(f"segid 0C and resid {x3_values[4]}")
            if alpha2_mu.resnames[0] != mtype.upper():
                print(f"Warning: Mutation {mutation_id} chain 0B not matching {mtype.upper()}")
            if alpha1_1_mu.resnames[0] != "GLY" or alpha1_2_mu.resnames[0] != "GLY":
                print(f"Warning: Mutation {mutation_id} chain 0A or 0C is not GLY")
        
        # 計算 mutant 的半徑 profile：對9個位置逐一計算
        radius_profile_mut = []  # mutant 的半徑 profile
        combined_center_list = []
        ca_positions = []
        for r1, r2, r3 in zip(x1_values, x2_values, x3_values):
            # 分別選取 mutant 結構中各鏈該位置的原子
            a1 = u1.select_atoms(f"segid 0A and resid {r1}")
            a2 = u1.select_atoms(f"segid 0B and resid {r2}")
            a3 = u1.select_atoms(f"segid 0C and resid {r3}")
            # print(f"Selections lengths: {len(a1)}, {len(a2)}, {len(a3)}") #會印出每個residue有幾個原子，若為0，表示超出這條chain的長度了，但是不會報錯
            # 分別計算各原子群的幾何中心。若沒也原子，則會算出一個空的矩陣
            if len(a1) == 0:
                print(f"⚠️ chain 0A empty at frame {t.frame}, resid {r1}")
                c1 = np.array([np.nan, np.nan, np.nan])
            else:
                c1 = a1.center_of_geometry()

            if len(a2) == 0:
                print(f"⚠️ chain 0B empty at frame {t.frame}, resid {r2}")
                c2 = np.array([np.nan, np.nan, np.nan])
            else:
                c2 = a2.center_of_geometry()

            if len(a3) == 0:
                print(f"⚠️ chain 0C empty at frame {t.frame}, resid {r3}")
                c3 = np.array([np.nan, np.nan, np.nan])
            else:
                c3 = a3.center_of_geometry()
            # 三條鏈的 Cα 坐標
            ca_positions.append(np.array([c1, c2, c3]))
            print(np.array(ca_positions).shape)
            # 合併三個原子群，計算整體幾何中心
            combined = AtomGroup(a1 + a2 + a3)
            combined_center = combined.center_of_geometry()
            combined_center_list.append(combined_center)
        combined_center_list = np.array(combined_center_list) #shape (9, 3) 9 positions，每個位置3D座標(x,y,z)
        ca_positions = np.array(ca_positions)  # shape (9, 3, 3) 9 positions，每個位置3條鏈的Cα的3D座標
        radius_profile_mut, axis_points = compute_collagen_radius(
            combined_center_list,
            ca_positions,
            smooth=0.5
        )
        # print(radius_profile_mut)
        radius_traj [t.frame] = radius_profile_mut
    radius_all.append(radius_traj)
radius_all = np.array (radius_all)

(1, 3, 3)
(2, 3, 3)
(3, 3, 3)
(4, 3, 3)
(5, 3, 3)
(6, 3, 3)
(7, 3, 3)
(8, 3, 3)
⚠️ chain 0B empty at frame 0, resid 2081
(9, 3, 3)
⚠️ Layer 8, Chain 1 is missing (skipped)
(1, 3, 3)
(2, 3, 3)
(3, 3, 3)
(4, 3, 3)
(5, 3, 3)
(6, 3, 3)
(7, 3, 3)
(8, 3, 3)
⚠️ chain 0B empty at frame 1, resid 2081
(9, 3, 3)
⚠️ Layer 8, Chain 1 is missing (skipped)
(1, 3, 3)
(2, 3, 3)
(3, 3, 3)
(4, 3, 3)
(5, 3, 3)
(6, 3, 3)
(7, 3, 3)
(8, 3, 3)
⚠️ chain 0B empty at frame 2, resid 2081
(9, 3, 3)
⚠️ Layer 8, Chain 1 is missing (skipped)
(1, 3, 3)
(2, 3, 3)
(3, 3, 3)
(4, 3, 3)
(5, 3, 3)
(6, 3, 3)
(7, 3, 3)
(8, 3, 3)
⚠️ chain 0B empty at frame 3, resid 2081
(9, 3, 3)
⚠️ Layer 8, Chain 1 is missing (skipped)
(1, 3, 3)
(2, 3, 3)
(3, 3, 3)
(4, 3, 3)
(5, 3, 3)
(6, 3, 3)
(7, 3, 3)
(8, 3, 3)
⚠️ chain 0B empty at frame 4, resid 2081
(9, 3, 3)
⚠️ Layer 8, Chain 1 is missing (skipped)
(1, 3, 3)
(2, 3, 3)
(3, 3, 3)
(4, 3, 3)
(5, 3, 3)
(6, 3, 3)
(7, 3, 3)
(8, 3, 3)
⚠️ chain 0B empty at frame 5, resid 2081
(9, 3, 3)
⚠️ Layer 

In [50]:
print(radius_all)  # (6, 100, 9) 6種突變，每種突變100個frame，每個frame 9個位置的radius
print(radius_all.shape)

[[[4.1140674  3.27195688 2.58860797 4.78387753 5.40431301 5.2246838
   4.85457999 4.0209511  7.51416394]
  [4.06152501 3.68406354 2.4758006  4.81555588 5.7376422  5.09541602
   5.35087471 3.87557872 7.17074325]
  [4.22543808 3.6294032  2.47001548 4.69180701 5.96433794 5.32383932
   4.69529683 3.87379177 6.85961265]
  [4.09191973 3.390174   2.71132235 4.65581136 5.87381358 5.36371581
   5.2549692  4.07813191 6.84237789]
  [4.16200559 3.49532349 2.43132505 4.83048784 5.67657491 5.48632463
   5.01277485 3.87618009 7.39522543]
  [4.01959658 3.62578894 2.57357876 4.74543536 5.79482313 5.04470162
   4.74951553 3.65748412 7.30014606]
  [4.08635409 3.49220055 2.37995381 4.60281332 5.48964545 5.23832747
   4.79362397 3.97050668 6.88637852]
  [4.16641763 3.33082    2.47705362 4.80500375 5.65243556 5.11933287
   5.31675801 4.12456859 6.88296282]
  [4.09003281 3.44704471 2.57216052 4.55391303 5.69383003 5.03746322
   5.1400421  4.04280816 7.27551516]
  [4.26307451 3.32125256 2.63947998 4.64878898 

In [3]:
print(radius_all.shape)  # (6, 100, 9) 6種突變，每種突變100個frame，每個frame 9個位置的radius

(587, 100, 9)


In [12]:
data_485 = radius_all[204]   # 取出第485筆資料
print(data_485.shape)        # (100, 9)
print(data_485)              # 顯示數據內容

(100, 9)
[[5.01181462 4.11289097 3.50584739 2.65806265 5.70179117 5.53929298
  4.7753162  6.06020322 5.723084  ]
 [4.74360483 3.98687721 3.65830328 3.12494761 5.49716217 5.65406169
  5.10586382 5.78365916 5.36020452]
 [5.00762112 4.09372736 3.61286477 2.86270725 5.30479159 5.65039886
  4.5484594  5.67327551 5.14206925]
 [4.98088676 4.12617152 3.62615866 2.73877062 5.38944673 5.63432621
  4.83273166 5.63269937 5.09100965]
 [4.91462519 3.98419783 3.61057822 2.89457922 5.23612418 5.57754214
  5.0516814  5.52850186 5.32740148]
 [4.87522913 4.06525626 3.54440679 3.24732408 5.32707342 5.79125562
  5.15526841 5.24018883 4.99998018]
 [4.7929321  3.96507595 3.64811167 3.13430479 5.4521504  5.67196341
  5.03151946 5.04907031 5.15053788]
 [4.8773296  3.99223057 3.53378777 2.9290752  5.31483206 5.71663536
  4.97096288 5.38309439 5.36129892]
 [4.96464011 4.11750986 3.83517967 3.17419785 5.47149274 5.60831707
  5.07593997 5.72059752 5.69644942]
 [4.84452081 4.13994889 3.63742424 2.93334414 5.6752997